In [1]:
import pandas as pd

raw_name = "data.csv"
daily_name = "daily.csv"
weekly_name = "weekly.csv"
monthly_name ="monthly.csv"

load_chunksize = 1_000_000

print('Setup complete!')

Setup complete!


In [2]:

results = []

for chunk in pd.read_csv(raw_name, chunksize=load_chunksize):

    chunk = chunk.dropna(subset=["onpromotion"])
    chunk["onpromotion"] = chunk["onpromotion"].astype(int)

    # weighted promotion = sales * promotion
    chunk["promo_weight"] = chunk["unit_sales"] * chunk["onpromotion"]

    grouped = chunk.groupby("date").agg(
        sales=("unit_sales", "sum"),
        promo_weight=("promo_weight", "sum")
    ).reset_index()

    results.append(grouped)

print('Dataset loaded!')

daily = pd.concat(results)
# aggregate across chunks
daily = daily.groupby("date").agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()
daily["promotion"] = daily["promo_weight"] / daily["sales"].replace(0, pd.NA)

daily = daily[["date", "sales", "promotion"]]

daily["date"] = pd.to_datetime(daily["date"])
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["month"] = daily["date"].dt.month
daily["year"] = daily["date"].dt.year

daily.to_csv(daily_name, index=False)

print('Daily aggregation saved!')

C:\Users\csg12\AppData\Local\Temp\ipykernel_16380\395919109.py:3: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(raw_name, chunksize=load_chunksize):


Dataset loaded!
Daily aggregation saved!


In [6]:
# --- WEEKLY AGGREGATION ---
weekly = daily.copy()

# Reconstruct promo_weight to avoid averaging ratios
weekly["promo_weight"] = weekly["promotion"] * weekly["sales"]

weekly = weekly.groupby(pd.Grouper(key="date", freq="W")).agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()

weekly["promotion"] = weekly["promo_weight"] / weekly["sales"]

weekly = weekly[["date", "sales", "promotion"]]
weekly["day_of_week"] = weekly["date"].dt.dayofweek
weekly["month"] = weekly["date"].dt.month
weekly["year"] = weekly["date"].dt.year

weekly.to_csv(weekly_name, index=False)
print('Weekly aggregation saved!')

# --- MONTHLY AGGREGATION ---
monthly = daily.copy()

monthly["promo_weight"] = monthly["promotion"] * monthly["sales"]

monthly = monthly.groupby(pd.Grouper(key="date", freq="ME")).agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()

monthly["promotion"] = monthly["promo_weight"] / monthly["sales"]

monthly = monthly[["date", "sales", "promotion"]]
monthly["day_of_week"] = monthly["date"].dt.dayofweek
monthly["month"] = monthly["date"].dt.month
monthly["year"] = monthly["date"].dt.year

monthly.to_csv(monthly_name, index=False)
print('Monthly aggregation saved!')


Weekly aggregation saved!
Monthly aggregation saved!


In [7]:
if "daily" not in globals():
    daily = pd.read_csv(daily_name)
    print('Daily aggregation loaded!')

if "weekly" not in globals():
    weekly = pd.read_csv(weekly_name)
    print('Weekly aggregation loaded!')

if "monthly" not in globals():
    monthly = pd.read_csv(monthly_name)
    print('Monthly aggregation loaded!')
